# Chicago crimes with tensorflow

In this project I will use the chicago crime dataset to skill up in Tensorflow. The idea is to predict if a given crime will lead to an arrest or not. 

Let's start by importing the relevant packages

In [ ]:
import os
import matplotlib
import numpy
import pandas
import seaborn
import tensorflow.keras as keras

## Importing data
First, let's import a subset of the data in a .csv file via pandas and make some exploratory analysis. This way I can  get a first understanding of the kind of pre-processing I need for the data.

In [ ]:
file_name = 'Crimes_-_2020.csv'
file_source_path = os.path.join(os.getcwd(), file_name)
chicago_crime_dataset = pandas.read_csv(file_source_path)

First of all, let's inspect a few entries of the dataset and the data types

In [ ]:
chicago_crime_dataset.head(5)

In [ ]:
chicago_crime_dataset.dtypes

This shows immediately some anomalies that do not make sense according to the type of data I expect. In particular:
1. Case number and Block are misnamed. Case number is an alphanumeric identifier, while Block is the address.
2. Date and Updated on are date objects and not strings. Moreover, the format is non-standard.
3. ID, Beat, District, Wards and Community areas are identification codes, not numbers.
4. Year is redundant. It's also not particularly interesting because this dataset covers 2020 only.
5. Location is redundant, at least for the specific purpoeses of this project. Furthermore, I checked that entries with missing values of XY coordinates do not have entries of latitude-longitude coordinates. The analysis is not shown here, but it can be performed with chicago_crime_dataset.loc[chicago_crime_dataset['Longitude'].isna(), ['X Coordinate', 'Y Coordinate', 'Latitude', 'Longitude']].head(10).sum()

For the moment I will correct point 2 to 5 only because I will need to stick with the original column namings when I will play around with tensorflow later on.




In [ ]:
# recast dates in correct iso formats
date_format = '%m/%d/%Y %I:%M:%S %p'
chicago_crime_dataset['Date'] = pandas.to_datetime(chicago_crime_dataset['Date'], format = date_format).head()
chicago_crime_dataset['Updated On'] = pandas.to_datetime(chicago_crime_dataset['Updated On'], format = date_format).head()

# recast faux integers as string identifiers
int_to_str_conversions = {'ID' : 'str',
                          'Beat' : 'str',
                          'District' : 'str',
                          'Ward' : 'str',
                          'Community Area' : 'str'}
chicago_crime_dataset = chicago_crime_dataset.astype(int_to_str_conversions)

# drop redundant columns
chicago_crime_dataset.pop('Year')
chicago_crime_dataset.pop('Location')
chicago_crime_dataset.pop('Longitude')
chicago_crime_dataset.pop('Latitude')

# show new data structure
chicago_crime_dataset.dtypes

I might need more adjustmens later on, but let's be happy with it for now.  As safety check, let's also count the number of nans to get an idea of how much information is missing

In [ ]:
# isolate columns with nans, and count the nans
nans_count = chicago_crime_dataset.isna().sum()
columns_with_nans = nans_count[nans_count > 0]

# inspect missing not at random by counting nans from rows where Date is missing
missing_not_at_random = chicago_crime_dataset.loc[chicago_crime_dataset['Date'].isna(), columns_with_nans.index].isna().sum()

print('Rows with nans:')
print(columns_with_nans)
print(' ')
print('Rows with nans on rows that have a missing Date')
print(missing_not_at_random)

There are a few things to keep in mind:
1. A relatively large number of cases do not have a Date. The number of cases without also coindiced with the number of cases without an update
2. Rows with missing Date can also have missing Coordinates, Location description and Updated On. I wonder if this arise because of some information embargo.



Let's now describe the dataframe content. The only continuous variables left are the XY coordinates.

In [ ]:
# quick overview of continuous variables
seaborn.scatterplot(x = 'X Coordinate', 
                    y = 'Y Coordinate',
                    hue = 'District', 
                    data = chicago_crime_dataset)

These data distributes according to the shape of Chicago. District, Beats and similar are spatially connected in a meaningful way (plot shown only for Districts). 

Speaking instead of cateforical variables:

In [ ]:
# Quick overview of categorical variables
chicago_crime_dataset.astype('object').describe()

In [ ]:
# inspect structure of cases with multiple crime IDs in it.
cases_with_multiple_IDs = chicago_crime_dataset.loc[chicago_crime_dataset\
                                                .duplicated(subset = 'Case Number', keep = False)]\
                                                .sort_values('Case Number')
cases_with_multiple_IDs.astype('object').describe()

The behavior of categorical observables require a little more care. Specifically, one can observe that:
1. In general, all entries tend to have an unique ID and unique Case Number with an exceptions. A bunch of homicide entries have the same Case Number but multiple ID and redacted Date (see cell below). I believe that these cases involve multiple homicides under information embargo, but I have no way to check it.
2. There are data only for 5 dates in this dataset. This is not surprising because the .csv file with the data is just a subsample of the total dataset. I need to remember this for further developments.

in general I would produce histograms that counts the categorical variables at this point. However, the sheer size of the dataset, and the number of categories in the categorical variables make this unfeasible.

I will recover this part later on with bigquery and its ability to aggregate from the Cloud. For the moment I will count only the 'Arrest' column because this is the label I want to predict. 


In [ ]:
seaborn.countplot(x = chicago_crime_dataset['Arrest'])

The Arrest thus have a strong bias towars crimes that do not lead to an arrest. Well, this is reasonable given that many crimes correspond to minor offences for which arrests do not make sense.

# Building the predictive model
W